In [ ]:
import sys
sys.path.append('..')

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from simulation.create_network import create_microgrid, add_nodes, connect_nodes
from simulation.visualization import plot_network

# Set random seed for reproducibility
np.random.seed(42)

## 1. Create Base Microgrid with Different Topologies

In [ ]:
# Create mesh topology
mesh_graph, mesh_meta = create_microgrid(num_nodes=8, topology='mesh', seed=42)
print(f"Mesh Network: {mesh_meta['num_nodes']} nodes, {mesh_meta['num_edges']} edges")
print(f"Edges: {len(mesh_meta['edges'])}")

In [ ]:
# Create ring topology
ring_graph, ring_meta = create_microgrid(num_nodes=8, topology='ring', seed=42)
print(f"Ring Network: {ring_meta['num_nodes']} nodes, {ring_meta['num_edges']} edges")

In [ ]:
# Create tree topology
tree_graph, tree_meta = create_microgrid(num_nodes=8, topology='tree', seed=42)
print(f"Tree Network: {tree_meta['num_nodes']} nodes, {tree_meta['num_edges']} edges")

## 2. Visualize Different Topologies

In [ ]:
# Plot mesh topology
fig, ax = plot_network(mesh_graph, title='Mesh Topology', layout='spring')
plt.show()

In [ ]:
# Plot ring topology
fig, ax = plot_network(ring_graph, title='Ring Topology', layout='circular')
plt.show()

In [ ]:
# Plot tree topology
fig, ax = plot_network(tree_graph, title='Tree Topology', layout='spring')
plt.show()

## 3. Modify Microgrid Structure

In [ ]:
# Start with a mesh topology
custom_graph, _ = create_microgrid(num_nodes=6, topology='ring', seed=42)
print(f"Initial network: {custom_graph.number_of_nodes()} nodes, {custom_graph.number_of_edges()} edges")

# Add new nodes
custom_graph = add_nodes(custom_graph, num_new_nodes=3)
print(f"After adding nodes: {custom_graph.number_of_nodes()} nodes, {custom_graph.number_of_edges()} edges")

In [ ]:
# Connect new nodes to existing network
custom_graph = connect_nodes(custom_graph, node1=6, node2=0, resistance=0.03, reactance=0.08, capacity=150)
custom_graph = connect_nodes(custom_graph, node1=7, node2=1, resistance=0.03, reactance=0.08, capacity=150)
custom_graph = connect_nodes(custom_graph, node1=8, node2=2, resistance=0.03, reactance=0.08, capacity=150)
print(f"After connecting new nodes: {custom_graph.number_of_nodes()} nodes, {custom_graph.number_of_edges()} edges")

In [ ]:
# Visualize modified network
fig, ax = plot_network(custom_graph, title='Modified Microgrid', layout='spring')
plt.show()

## 4. Analyze Network Properties

In [ ]:
# Compute network statistics
for name, graph in [('Mesh', mesh_graph), ('Ring', ring_graph), ('Tree', tree_graph)]:
    print(f"\n{name} Topology:")
    print(f"  Nodes: {graph.number_of_nodes()}")
    print(f"  Edges: {graph.number_of_edges()}")
    print(f"  Density: {nx.density(graph):.3f}")
    print(f"  Connected: {nx.is_connected(graph)}")
    print(f"  Average degree: {sum(dict(graph.degree()).values()) / graph.number_of_nodes():.2f}")

In [ ]:
# Analyze node attributes
print("\nNode Attributes (Mesh Topology):")
for node in list(mesh_graph.nodes())[:3]:
    attrs = mesh_graph.nodes[node]
    print(f"Node {node}:")
    for key, value in attrs.items():
        print(f"  {key}: {value:.3f}")

In [ ]:
# Analyze edge attributes
print("\nEdge Attributes (Mesh Topology):")
for edge in list(mesh_graph.edges())[:3]:
    attrs = mesh_graph.edges[edge]
    print(f"Edge {edge}:")
    for key, value in attrs.items():
        print(f"  {key}: {value:.3f}")

## 5. Visualize Comparison

In [ ]:
# Compare three topologies side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

graphs = [('Mesh', mesh_graph), ('Ring', ring_graph), ('Tree', tree_graph)]

for idx, (name, graph) in enumerate(graphs):
    ax = axes[idx]
    
    # Choose layout
    if name == 'Ring':
        pos = nx.circular_layout(graph)
    else:
        pos = nx.spring_layout(graph, k=2, iterations=50, seed=42)
    
    # Node colors
    node_colors = []
    for node in graph.nodes():
        gen = graph.nodes[node].get('generation', 0)
        if gen > 50:
            node_colors.append('red')
        elif gen > 0:
            node_colors.append('orange')
        else:
            node_colors.append('lightblue')
    
    # Draw
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=400, ax=ax, alpha=0.8)
    nx.draw_networkx_edges(graph, pos, width=2, ax=ax, alpha=0.6)
    nx.draw_networkx_labels(graph, pos, font_size=8, ax=ax)
    
    ax.set_title(f'{name} ({graph.number_of_nodes()} nodes)', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()